# Gold — Category Growth Streaks

Find product categories with **3+ consecutive calendar months** of positive month-over-month revenue growth on delivered orders.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.gold.category_growth as growth_module
importlib.reload(growth_module)

from src.gold.category_growth import (
    GoldCategoryGrowthConfig,
    build_category_growth_streaks,
    MIN_STREAK_LENGTH,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = GoldCategoryGrowthConfig()
print("Loaded from:", growth_module.__file__)
print("Minimum streak length:", MIN_STREAK_LENGTH)

In [ ]:
streaks = build_category_growth_streaks(spark, config)
streaks.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Qualifying streaks:", spark.table(config.target_table).count())
display(spark.table(config.target_table))

In [ ]:
import json

written = spark.table(config.target_table)
rows = [row.asDict() for row in written.collect()]

report = {
    "task": "gold_category_growth_streaks",
    "target_table": config.target_table,
    "min_streak_length": MIN_STREAK_LENGTH,
    "qualifying_streak_count": len(rows),
    "qualifying_streaks": rows,
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/gold_category_growth.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)